# Part 2A-ii: Custom Dropout — MC Alpha Dropout

**Objective:** Implement MCAlphaDropout for SELU-based self-normalizing networks.

---

In [ ]:
import numpy as np, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import torch, torch.nn as nn

(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()
X_train, X_test = X_train.astype("float32")/255.0, X_test.astype("float32")/255.0
y_train, y_test = y_train.flatten(), y_test.flatten()
X_flat_tr, X_flat_te = X_train.reshape(len(X_train),-1), X_test.reshape(len(X_test),-1)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


In [ ]:
# TF: MCAlphaDropout — always active (for MC inference)
class MCAlphaDropout(layers.Layer):
    """Alpha Dropout that stays active during inference for MC estimation.
    Designed for SELU networks — preserves self-normalizing property.
    """
    def __init__(self, rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.rate = rate

    def call(self, inputs, training=None):
        # KEY: always apply dropout (ignore training flag) for MC inference
        if self.rate == 0:
            return inputs
        alpha = 1.6732632423543772
        scale = 1.0507009873554805
        alpha_p = -alpha * scale
        kept = tf.cast(tf.random.uniform(tf.shape(inputs)) >= self.rate, inputs.dtype)
        a = ((1 - self.rate) * (1 + self.rate * alpha_p**2))**-0.5
        b = -a * alpha_p * self.rate
        x = inputs * kept + alpha_p * (1 - kept)
        return a * x + b

# Self-normalizing network with MCAlphaDropout
model = keras.Sequential([
    layers.Input(shape=(3072,)),
    layers.Dense(256, activation='selu', kernel_initializer='lecun_normal'),
    MCAlphaDropout(0.1),
    layers.Dense(128, activation='selu', kernel_initializer='lecun_normal'),
    MCAlphaDropout(0.1),
    layers.Dense(64, activation='selu', kernel_initializer='lecun_normal'),
    MCAlphaDropout(0.1),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(X_flat_tr, y_train, epochs=20, batch_size=256, validation_split=0.2, verbose=1)

# MC inference — 100 forward passes (dropout stays active automatically)
sample = X_flat_te[:10]
mc_preds = np.array([model(sample).numpy() for _ in range(100)])  # (100, 10, 10)
mean_preds = mc_preds.mean(axis=0)
std_preds = mc_preds.std(axis=0)

class_names = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
print("\nMC Alpha Dropout Predictions:")
for i in range(5):
    pred = np.argmax(mean_preds[i])
    print(f"  Sample {i}: {class_names[pred]} (conf={mean_preds[i][pred]:.3f} ± {std_preds[i][pred]:.3f}), True: {class_names[y_test[i]]}")

Epoch 1/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 8s 33ms/step - accuracy: 0.2440 - loss: 2.1226 - val_accuracy: 0.3048 - val_loss: 1.9140
Epoch 2/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.3336 - loss: 1.8453 - val_accuracy: 0.3474 - val_loss: 1.8044
Epoch 3/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.3609 - loss: 1.7686 - val_accuracy: 0.3744 - val_loss: 1.7601
Epoch 4/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.3826 - loss: 1.7164 - val_accuracy: 0.3810 - val_loss: 1.7221
Epoch 5/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4013 - loss: 1.6691 - val_accuracy: 0.3997 - val_loss: 1.7019
Epoch 6/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4147 - loss: 1.6345 - val_accuracy: 0.4016 - val_loss: 1.6832
Epoch 7/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4262 - loss: 1.6030 - val_accuracy: 0.4070 - val_loss: 1.6811
Epoch 8/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4383 - loss: 1.5718 - val_accuracy: 0

In [ ]:
# PyTorch equivalent
class MCAlphaDropoutPT(nn.Module):
    """PyTorch MCAlphaDropout — always active."""
    def __init__(self, rate=0.1):
        super().__init__()
        self.rate = rate

    def forward(self, x):
        if self.rate == 0: return x
        alpha = 1.6732632423543772
        scale = 1.0507009873554805
        alpha_p = -alpha * scale
        kept = (torch.rand_like(x) >= self.rate).float()
        a = ((1 - self.rate) * (1 + self.rate * alpha_p**2))**-0.5
        b = -a * alpha_p * self.rate
        return a * (x * kept + alpha_p * (1 - kept)) + b

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pt_model = nn.Sequential(
    nn.Linear(3072, 256), nn.SELU(), MCAlphaDropoutPT(0.1),
    nn.Linear(256, 128), nn.SELU(), MCAlphaDropoutPT(0.1),
    nn.Linear(128, 10)
).to(device)

# Initialize with LeCun Normal for SELU
for m in pt_model:
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='linear')

opt = torch.optim.Adam(pt_model.parameters(), lr=1e-3)
loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(torch.FloatTensor(X_flat_tr), torch.LongTensor(y_train)),
    batch_size=256, shuffle=True)
crit = nn.CrossEntropyLoss()
for ep in range(20):
    pt_model.train()
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(); crit(pt_model(xb), yb).backward(); opt.step()
# MC inference
sample_t = torch.FloatTensor(X_flat_te[:10]).to(device)
with torch.no_grad():
    preds = torch.stack([torch.softmax(pt_model(sample_t), dim=1) for _ in range(100)])
print(f"\nPyTorch MCAlphaDropout — Mean conf of top class: {preds.mean(0).max(1).values.mean():.3f}")
print(f"Mean uncertainty: {preds.std(0).mean():.4f}")


PyTorch MCAlphaDropout — Mean conf of top class: 0.461
Mean uncertainty: 0.0367


## Key Takeaways
- MCAlphaDropout = Alpha Dropout that stays active during inference
- Use with SELU activation + LeCun initialization for self-normalizing networks
- Enables uncertainty estimation without modifying the training procedure
- Higher uncertainty → model is less confident → flag for human review